In [1]:
import random
from collections import deque, namedtuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import HTML
from matplotlib import animation

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### Python refresher: subclassing `nn.Module`
Every PyTorch network is a class that inherits from `nn.Module` and defines a `forward()` method describing how input data flows through it. This is the same "class with methods" pattern you'd use for any custom Python object; PyTorch just expects specific method names (`__init__` and `forward`) so it knows how to run and train the network.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden_size=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_actions)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "done"])

class ReplayBuffer: 
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append(Transition(state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch)) # transpose list of tuples into tuples of lists

    def __len__(self):
        return len(self.buffer)

In [2]:
class DQNAgent:
    def __init__(self, obs_dim, n_actions, hidden_size=128, lr=1e-3, gamma=0.99,
                 buffer_capacity=50000, batch_size=64):
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size

        self.q_network = QNetwork(obs_dim, n_actions, hidden_size)
        self.target_network = QNetwork(obs_dim, n_actions, hidden_size)
        self.sync_target()
        self.target_network.eval()  # target network is never trained directly

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_capacity)

    def sync_target(self):
        """Copy the Q-network's weights into the target network."""
        self.target_network.load_state_dict(self.q_network.state_dict())

    def select_action(self, state, epsilon):
        if random.random() < epsilon:
            return random.randrange(self.n_actions)  # explore
        with torch.no_grad():
            state_t = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
            q_values = self.q_network(state_t)
        return int(torch.argmax(q_values, dim=1).item())  # exploit

    def store(self, state, action, reward, next_state, done):
        self.buffer.push(state, action, reward, next_state, done)

    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return None # not enough experience yet

        batch = self.buffer.sample(self.batch_size)

        states = torch.as_tensor(np.array(batch.state), dtype=torch.float32)
        actions = torch.as_tensor(batch.action, dtype=torch.int64).unsqueeze(1)
        rewards = torch.as_tensor(batch.reward, dtype=torch.float32).unsqueeze(1)
        next_states = torch.as_tensor(np.array(batch.next_state), dtype=torch.float32)
        dones = torch.as_tensor(batch.done, dtype=torch.float32).unsqueeze(1)

        # Q-values the network currently predicts for the actions actually taken
        q_values = self.q_network(states).gather(1, actions)

        # Q-learning target, using the target network for stability
        with torch.no_grad():
            next_q_values = self.target_network(next_states).max(dim=1, keepdim=True)[0]
            td_target = rewards + self.gamma * next_q_values * (1.0 - dones)

        loss = nn.functional.mse_loss(q_values, td_target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()